# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook generates the actionable prioritization queue and operational playbook for **Lane 2: Refresh / Content Opportunity Scoring**.

> Skill loaded: `writing-honest-claims`.

## 1. Ranked actions + reason codes

### ML-Scored Action Queue
Using our trained Gradient Boosting model, we score all 30,000 candidate content items, assign interpretable reason codes and action labels, and export the prioritized queue.

In [1]:
# Section 1: Ranked Actions Queue & Reason Code Assignment
import os
import json
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Feature setup
num_cols = [
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'engaged_sessions_90d',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'content_age_days',
    'days_since_last_update', 'search_volume', 'cpc', 'competition', 'word_count', 'char_count'
]
df['has_word_count'] = (~df['word_count'].isna()).astype(int)
df['has_search_volume'] = (~df['search_volume'].isna()).astype(int)
df['has_cpc'] = (~df['cpc'].isna()).astype(int)
df['has_pos_data'] = (df['avg_position'] > 0).astype(int)
df['is_striking'] = (df['position_tier'] == 'striking').astype(int)
df['is_peak_decay_age'] = ((df['days_since_last_update'] >= 90) & (df['days_since_last_update'] <= 180)).astype(int)
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])

num_cols += ['has_word_count', 'has_search_volume', 'has_cpc', 'has_pos_data', 'is_striking', 'is_peak_decay_age', 'log_impressions_90d']
cat_cols = ['content_type', 'main_intent', 'position_tier', 'freshness_tier', 'impression_tier']
all_features = num_cols + cat_cols

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

pipe = Pipeline([('preprocessor', preprocessor), ('classifier', HistGradientBoostingClassifier(max_iter=100, max_depth=5, random_state=42))])
pipe.fit(df[all_features], df['is_declining_label'])
df['ml_opportunity_score'] = pipe.predict_proba(df[all_features])[:, 1]

# Reason Code & Action Assignment
def assign_reason_code(row):
    if row['position_tier'] == 'striking' and (90 <= row['days_since_last_update'] <= 180):
        return 'STRIKING_POSITION_PEAK_DECAY'
    elif row['position_tier'] == 'striking':
        return 'STRIKING_POSITION_OPPORTUNITY'
    elif (90 <= row['days_since_last_update'] <= 180):
        return 'PEAK_DECAY_AGE_WINDOW'
    elif row['position_tier'] == 'page_1':
        return 'PAGE_1_TRAFFIC_PROTECTION'
    else:
        return 'LOW_PRIORITY_STABLE'

def assign_action_label(row):
    if row['position_tier'] == 'striking' and (90 <= row['days_since_last_update'] <= 180):
        return 'REFRESH_AND_EXPAND_KEYWORDS'
    elif row['position_tier'] == 'striking':
        return 'OPTIMIZE_ON_PAGE_SEO'
    elif (90 <= row['days_since_last_update'] <= 180):
        return 'UPDATE_OUTDATED_SECTIONS'
    elif row['position_tier'] == 'page_1':
        return 'INTERNAL_LINK_BOOST'
    else:
        return 'MONITOR_ONLY'

df['reason_code'] = df.apply(assign_reason_code, axis=1)
df['action_label'] = df.apply(assign_action_label, axis=1)

ranked_df = df.sort_values(by=['ml_opportunity_score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)
ranked_df['ml_rank'] = np.arange(1, len(ranked_df) + 1)

print("=== TOP 10 RANKED OPPORTUNITY QUEUE (ML MODEL) ===")
cols_show = ['ml_rank', 'content_id', 'client_id', 'ml_opportunity_score', 'reason_code', 'action_label', 'impressions_90d', 'avg_position', 'is_declining_label']
print(ranked_df.head(10)[cols_show].to_string(index=False))


=== TOP 10 RANKED OPPORTUNITY QUEUE (ML MODEL) ===
 ml_rank           content_id         client_id  ml_opportunity_score                   reason_code             action_label  impressions_90d  avg_position  is_declining_label
       1 content_ea733ea77a8f client_3fdba35f04              0.963640     PAGE_1_TRAFFIC_PROTECTION      INTERNAL_LINK_BOOST              397           4.1                   1
       2 content_9b6df29f7889 client_3fdba35f04              0.956738         PEAK_DECAY_AGE_WINDOW UPDATE_OUTDATED_SECTIONS             1622           3.1                   1
       3 content_ab82c4705992 client_7f2253d7e2              0.955008           LOW_PRIORITY_STABLE             MONITOR_ONLY            12315          20.6                   1
       4 content_d939da189d28 client_7f2253d7e2              0.953973 STRIKING_POSITION_OPPORTUNITY     OPTIMIZE_ON_PAGE_SEO             8915          18.6                   1
       5 content_53d94816b5b0 client_7f2253d7e2              0.953559

## 2. Intended use and limits

### Intended Operational Scope
Provides weekly decision-support priority scoring for editorial teams.

In [2]:
# Section 2: Intended Use & Boundaries
print("=== INTENDED USE & OPERATIONAL BOUNDARIES ===")
print("1. Target Users: Editorial leads, SEO strategists, content managers.")
print("2. Operational Cadence: Weekly candidate refresh queue prioritization.")
print("3. Limits: Does not predict Google algorithm changes; does not guarantee ranking recovery.")


=== INTENDED USE & OPERATIONAL BOUNDARIES ===
1. Target Users: Editorial leads, SEO strategists, content managers.
2. Operational Cadence: Weekly candidate refresh queue prioritization.
3. Limits: Does not predict Google algorithm changes; does not guarantee ranking recovery.


## 3. Human review + the no-go list

### Human Pre-Execution Checklist & No-Go Boundaries
Mandatory safety checks before approving content rewrite actions.

In [3]:
# Section 3: Human Review & No-Go Automation List
print("=== HUMAN REVIEW CHECKLIST & NO-GO AUTOMATION LIST ===")
print("Human Review Mandatory Checks:")
print("  [x] Check if high-volume page traffic is driven by branded search terms.")
print("  [x] Verify whether content update was recently published on staging.")
print("  [x] Confirm search intent hasn't shifted to video/interactive tools.")
print("\nNo-Go Automation List:")
print("  [!] NEVER automatically overwrite top-performing Page 1 articles without human editor sign-off.")
print("  [!] NEVER trigger bulk auto-generated AI text rewrites on high-authority client domains.")


=== HUMAN REVIEW CHECKLIST & NO-GO AUTOMATION LIST ===
Human Review Mandatory Checks:
  [x] Check if high-volume page traffic is driven by branded search terms.
  [x] Verify whether content update was recently published on staging.
  [x] Confirm search intent hasn't shifted to video/interactive tools.

No-Go Automation List:
  [!] NEVER automatically overwrite top-performing Page 1 articles without human editor sign-off.
  [!] NEVER trigger bulk auto-generated AI text rewrites on high-authority client domains.


## 4. Monitoring / retrain triggers

### Model Health Monitoring Triggers
Triggers for model retraining and performance re-evaluation.

In [4]:
# Section 4: Monitoring & Retrain Triggers
print("=== MONITORING & RETRAIN TRIGGERS ===")
print("1. Model Performance Drift: Retrain if P@50 on new client batches drops below 65.0%.")
print("2. Google Core Algorithm Update: Trigger immediate retraining following major Google Search core update releases.")
print("3. Data Distribution Drift: Retrain if >20% of client portfolio introduces new content types.")


=== MONITORING & RETRAIN TRIGGERS ===
1. Model Performance Drift: Retrain if P@50 on new client batches drops below 65.0%.
2. Google Core Algorithm Update: Trigger immediate retraining following major Google Search core update releases.
3. Data Distribution Drift: Retrain if >20% of client portfolio introduces new content types.


## 5. Exports for the paper

### Export Playbook Files
Writing `work/outputs/ml_action_playbook.csv` and `work/outputs/ml_metrics.json`.

In [5]:
# Section 5: Export Playbook Queue & Metrics
os.makedirs("work/outputs", exist_ok=True)
output_cols = ['ml_rank', 'content_id', 'client_id', 'ml_opportunity_score', 'reason_code', 'action_label', 'impressions_90d', 'avg_position', 'days_since_last_update']
ranked_df[output_cols].to_csv("work/outputs/ml_action_playbook.csv", index=False)

metrics = {
    "model_name": "gradient_boosting_opportunity_scorer_v1",
    "total_scored_items": len(ranked_df),
    "top_50_declining_precision": float(ranked_df.head(50)['is_declining_label'].mean()),
    "top_100_declining_precision": float(ranked_df.head(100)['is_declining_label'].mean())
}
with open("work/outputs/ml_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Successfully exported work/outputs/ml_action_playbook.csv ({len(ranked_df):,} rows)")
print("Saved ML metrics receipt to work/outputs/ml_metrics.json")


Successfully exported work/outputs/ml_action_playbook.csv (30,000 rows)
Saved ML metrics receipt to work/outputs/ml_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.